In [ ]:
  for x in [df, dff]:
        assert isinstance(x, pd.core.frame.DataFrame)
    assert isinstance(roundn, int)
    pass

    dfb = pd.concat([df.copy().drop(columns=["ctrls"]), dff.copy()])
    dfb.rename(
        columns={
            "treatbeta": "1beta",
            "stderr": "2stderr",
            "meanctrl": "3mean",
            "N": "4N",
        },
        inplace=True,
    )

    # get stars
    dfb["stars"] = 0
    dfb.loc[(abs(dfb["1beta"]) - dfb["2stderr"] * 1.645) > 0, "stars"] = 1
    dfb.loc[(abs(dfb["1beta"]) - dfb["2stderr"] * 1.96) > 0, "stars"] = 2
    dfb.loc[(abs(dfb["1beta"]) - dfb["2stderr"] * 2.576) > 0, "stars"] = 3

    # stringify stats
    dfb["1beta"] = dfb["1beta"].apply(lambda x: "{n:.{d}f}".format(d=str(roundn), n=x))
    # dfb['1_beta'] = dfb.apply(lambda x: '{:.2f}'.format(x['1_beta']) \
    #                           if x['depvar'][:6] != 'videos' \
    #                           else '{:.2f}'.format(x['1_beta']), 1)
    dfb["1beta"] = dfb["1beta"] + dfb.stars.apply(lambda x: "*" * x)
    dfb["2stderr"] = dfb["2stderr"].apply(
        lambda x: "({n:.{d}f})".format(d=str(roundn), n=x)
    )
    dfb["3mean"] = dfb["3mean"].apply(lambda x: "{n:.{d}f}".format(d=str(roundn), n=x))
    dfb["4N"] = dfb["4N"].apply(lambda x: str(int(x)))
    dfb = dfb.drop(columns=["stars"])

    # reshape long (stack beta, stderr, mean, N)
    dfb = dfb.melt(id_vars=["depvar", "model"]).sort_values(
        ["depvar", "model", "variable"]
    )
    dfb.rename(columns={"variable": "stat"}, inplace=True)
    # display(dfb.head())

    # reshape wide (side by side models)
    dfb = dfb.pivot(index=["depvar", "stat"], columns="model").reset_index()
    dfb.columns = dfb.columns.droplevel(0)
    dfb.columns = ["depvar", "stat", "FEs", "Neyman", "ITT", "All"]
    dfb = dfb[["depvar", "stat", "ITT", "Neyman", "All", "FEs"]]

    # replace depvar with actual names
    dfb["og"] = dfb.depvar.copy()
    dfb["depvar"] = dfb.depvar.apply(lambda x: name_dict.get(x))

    # Get one mean per depvar
    means = dfb.loc[dfb.stat == "3mean", ["og", "ITT"]]
    means.columns = ["og", "ctrlmean"]
    dfb = means.merge(dfb, on="og", how="inner")
    return dfb
